# TREC-COVID — BM25 + RM3 with Cross-Validation
Tunes BM25 (`b`, `k1`) and RM3 (`fb_terms`, `fb_docs`, `fb_lambda`) via 5-fold CV.
Query verbosity levels: **title**, **title+desc**, **title+desc+narr**.

## 1. Imports

In [3]:
import os
import re
from pathlib import Path

import pyterrier as pt
import pandas as pd
import ir_datasets
import nltk
import numpy as np
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from sklearn.model_selection import KFold

nltk.download('stopwords', quiet=True)

stemmer   = PorterStemmer()
stop_words = set(stopwords.words('english'))

print('All imports OK')

All imports OK


## 2. Dataset

In [5]:
dataset_name = 'irds:cord19/trec-covid'
dataset      = pt.get_dataset(dataset_name)
irds_dataset = ir_datasets.load('cord19/trec-covid')

print(f'Loaded: {dataset_name}')

Loaded: irds:cord19/trec-covid


## 3. Index
Reuses the same index built by `experiment.ipynb` (`index/trec_covid_index`).

In [7]:
pt.java.init()

index_dir        = str(Path('index/trec_covid_index').resolve())
index_properties = os.path.join(index_dir, 'data.properties')

if os.path.exists(index_properties):
    print('Found existing index, loading...')
    index_ref = pt.IndexRef.of(index_properties)
else:
    print('Building index...')
    os.makedirs(index_dir, exist_ok=True)

    seen = set()
    def doc_iter():
        for doc in irds_dataset.docs_iter():
            if doc.doc_id in seen:
                continue
            seen.add(doc.doc_id)
            body = doc.body if hasattr(doc, 'body') and doc.body else ''
            yield {
                'docno': doc.doc_id,
                'text':  (doc.title or '') + ' ' + (doc.abstract or '') + ' ' + body
            }

    indexer   = pt.IterDictIndexer(index_dir, overwrite=True,
                                   meta={'docno': 26, 'text': 16384})
    index_ref = indexer.index(doc_iter())
    print('Index built.')

index = pt.IndexFactory.of(index_ref)
print(f'Documents: {index.getCollectionStatistics().getNumberOfDocuments():,}')

Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


Building index...
04:18:08.021 [main] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (8is9x9sc) - further warnings are suppressed
04:23:24.490 [main] WARN org.terrier.structures.indexing.Indexer -- Indexed 59 empty documents
Index built.
Documents: 191,175


## 4. Queries & Preprocessing
Uses the same Porter stemmer + NLTK stopwords as `data_exploration.ipynb`.

In [9]:
topics_data = []
for query in irds_dataset.queries_iter():
    topics_data.append({
        'qid':         query.query_id,
        'title':       query.title,
        'description': query.description,
        'narrative':   query.narrative,
    })

df_topics = pd.DataFrame(topics_data)
print(f'Queries: {len(df_topics)}')

Queries: 50


In [10]:
def preprocess_text(text):
    """Lowercase, remove punctuation, Porter-stem, remove NLTK stopwords.
    Consistent with data_exploration.ipynb preprocessing (level 1, no spell-check)."""
    if pd.isna(text) or not text:
        return ''
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    tokens = [stemmer.stem(t) for t in text.split() if t not in stop_words]
    return ' '.join(tokens)

df_topics['query_title'] = df_topics['title'].apply(preprocess_text)

df_topics['query_title_desc'] = (
    df_topics['title'].fillna('') + ' ' + df_topics['description'].fillna('')
).apply(preprocess_text)

df_topics['query_title_desc_narr'] = (
    df_topics['title'].fillna('')    + ' '
    + df_topics['description'].fillna('') + ' '
    + df_topics['narrative'].fillna('')
).apply(preprocess_text)

print('Preprocessing done.')
df_topics[['qid','query_title','query_title_desc','query_title_desc_narr']].head(3)

Preprocessing done.


,qid,query_title,query_title_desc,query_title_desc_narr
0,1,coronaviru origin,coronaviru origin origin covid19,coronaviru origin origin covid19 seek rang inf...
1,2,coronaviru respons weather chang,coronaviru respons weather chang coronaviru re...,coronaviru respons weather chang coronaviru re...
2,3,coronaviru immun,coronaviru immun sarscov2 infect peopl develop...,coronaviru immun sarscov2 infect peopl develop...


In [11]:
verbosity_levels = {
    'title':           df_topics[['qid','query_title']].rename(columns={'query_title':'query'}),
    'title_desc':      df_topics[['qid','query_title_desc']].rename(columns={'query_title_desc':'query'}),
    'title_desc_narr': df_topics[['qid','query_title_desc_narr']].rename(columns={'query_title_desc_narr':'query'}),
}
print('Verbosity levels:', list(verbosity_levels.keys()))

Verbosity levels: ['title', 'title_desc', 'title_desc_narr']


## 5. Eval Setup

In [13]:
qrels        = dataset.get_qrels()
eval_metrics = ['map', 'ndcg_cut_10', 'P_10', 'recall_1000']
bm25         = pt.terrier.Retriever(index, wmodel='BM25', num_results=1000)

## 7. BM25 Parameter Tuning — 5-Fold Cross-Validation

Grid search over `b` and `k_1` using **5-fold CV** to avoid overfitting on the 50 queries.
- Queries are split into 5 folds of 10.
- For each candidate `(b, k1)`: train on 4 folds (40 queries) → pick params → evaluate on held-out fold (10 queries).
- The parameter combination with the best **average held-out MAP** across all folds is selected.
- Final retrievers are built with those parameters and re-evaluated on **all 50 queries** for the main results table.

In [15]:
from sklearn.model_selection import KFold
import numpy as np

b_values  = [0.3, 0.5, 0.75, 0.9]
k1_values = [0.5, 1.0, 1.2, 1.5, 2.0]
N_FOLDS   = 5

best_bm25_params = {}
tuning_records   = []

for level, topics_df in verbosity_levels.items():
    print(f'\nCV-tuning BM25 for verbosity level: {level}')

    qids   = topics_df['qid'].values
    kf     = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    splits = list(kf.split(qids))          # list of (train_idx, val_idx)

    param_cv_scores = {}  # (b, k1) -> list of per-fold MAP

    for b in b_values:
        for k1 in k1_values:
            fold_maps = []
            for train_idx, val_idx in splits:
                val_qids      = qids[val_idx]
                val_topics    = topics_df[topics_df['qid'].isin(val_qids)]
                # Filter qrels to validation queries only
                val_qrels     = qrels[qrels['qid'].isin(val_qids)]

                retriever = pt.terrier.Retriever(
                    index, wmodel='BM25', num_results=1000,
                    controls={'bm25.b': b, 'bm25.k_1': k1}
                )
                res = pt.Experiment(
                    retr_systems=[retriever],
                    topics=val_topics,
                    qrels=val_qrels,
                    eval_metrics=['map'],
                    names=[f'b={b}_k1={k1}']
                )
                fold_maps.append(res['map'].values[0])

            mean_map = float(np.mean(fold_maps))
            std_map  = float(np.std(fold_maps))
            param_cv_scores[(b, k1)] = mean_map
            tuning_records.append({
                'level': level, 'b': b, 'k1': k1,
                'cv_mean_map': mean_map, 'cv_std_map': std_map
            })

    best_b, best_k1 = max(param_cv_scores, key=param_cv_scores.get)
    best_bm25_params[level] = {'b': best_b, 'k1': best_k1}
    print(f'  Best CV params for {level}: b={best_b}, k1={best_k1} '
          f'-> mean held-out MAP={param_cv_scores[(best_b, best_k1)]:.4f}')

df_tuning = pd.DataFrame(tuning_records)
print('\nFull BM25 CV tuning results (sorted by level, mean MAP):')
display(df_tuning.sort_values(['level', 'cv_mean_map'], ascending=[True, False]))



CV-tuning BM25 for verbosity level: title
  Best CV params for title: b=0.5, k1=1.2 -> mean held-out MAP=0.1787

CV-tuning BM25 for verbosity level: title_desc
  Best CV params for title_desc: b=0.5, k1=1.5 -> mean held-out MAP=0.1962

CV-tuning BM25 for verbosity level: title_desc_narr
  Best CV params for title_desc_narr: b=0.75, k1=1.5 -> mean held-out MAP=0.2023

Full BM25 CV tuning results (sorted by level, mean MAP):


,level,b,k1,cv_mean_map,cv_std_map
7,title,0.50,1.2,0.178687,0.030695
8,title,0.50,1.5,0.178450,0.031860
2,title,0.30,1.2,0.177871,0.031864
6,title,0.50,1.0,0.177646,0.029724
3,title,0.30,1.5,0.177642,0.032621
9,title,0.50,2.0,0.177450,0.032005
1,title,0.30,1.0,0.177089,0.030445
4,title,0.30,2.0,0.176237,0.032734
12,title,0.75,1.2,0.170642,0.028635
5,title,0.50,0.5,0.170485,0.022717


## 8. Tuned BM25 Retrievers

In [17]:
tuned_bm25 = {}
for level, params in best_bm25_params.items():
    tuned_bm25[level] = pt.terrier.Retriever(
        index,
        wmodel='BM25',
        num_results=1000,
        controls={'bm25.b': params['b'], 'bm25.k_1': params['k1']}
    )
    print(f'Tuned BM25 [{level}]: b={params["b"]}, k1={params["k1"]}')

Tuned BM25 [title]: b=0.5, k1=1.2
Tuned BM25 [title_desc]: b=0.5, k1=1.5
Tuned BM25 [title_desc_narr]: b=0.75, k1=1.5


## 9. RM3 Query Expansion Tuning — 5-Fold Cross-Validation

RM3 parameters tuned with the **same 5 folds** as BM25 (same random_state=42 split),
so the held-out queries are consistent across both tuning stages.
- `fb_terms`: expansion term count
- `fb_docs`: pseudo-relevance feedback document count
- `fb_lambda`: weight of the PRF model (1 - fb_lambda = weight of original query)


In [19]:
fb_terms_values    = [5, 10, 20]
fb_docs_values     = [3, 5, 10]
orig_weight_values = [0.3, 0.5, 0.7]
N_FOLDS            = 5

best_rm3_params    = {}
rm3_tuning_records = []

for level, topics_df in verbosity_levels.items():
    print(f'\nCV-tuning RM3 for verbosity level: {level}')

    qids   = topics_df['qid'].values
    kf     = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)  # same seed as BM25
    splits = list(kf.split(qids))

    base        = tuned_bm25[level]
    bm25_params = best_bm25_params[level]

    param_cv_scores = {}  # (fb_terms, fb_docs, orig_w) -> mean held-out MAP

    for fb_terms in fb_terms_values:
        for fb_docs in fb_docs_values:
            for orig_w in orig_weight_values:
                fold_maps = []
                for train_idx, val_idx in splits:
                    val_qids   = qids[val_idx]
                    val_topics = topics_df[topics_df['qid'].isin(val_qids)]
                    val_qrels  = qrels[qrels['qid'].isin(val_qids)]

                    rm3  = pt.rewrite.RM3(
                        index, fb_terms=fb_terms,
                        fb_docs=fb_docs, fb_lambda=orig_w
                    )
                    pipe = base >> rm3 >> pt.terrier.Retriever(
                        index, wmodel='BM25', num_results=1000,
                        controls={'bm25.b': bm25_params['b'],
                                  'bm25.k_1': bm25_params['k1']}
                    )
                    res = pt.Experiment(
                        retr_systems=[pipe],
                        topics=val_topics,
                        qrels=val_qrels,
                        eval_metrics=['map'],
                        names=[f'rm3_t{fb_terms}_d{fb_docs}_w{orig_w}']
                    )
                    fold_maps.append(res['map'].values[0])

                mean_map = float(np.mean(fold_maps))
                std_map  = float(np.std(fold_maps))
                param_cv_scores[(fb_terms, fb_docs, orig_w)] = mean_map
                rm3_tuning_records.append({
                    'level': level,
                    'fb_terms': fb_terms, 'fb_docs': fb_docs, 'fb_lambda': orig_w,
                    'cv_mean_map': mean_map, 'cv_std_map': std_map
                })

    best_t, best_d, best_w = max(param_cv_scores, key=param_cv_scores.get)
    best_rm3_params[level] = {'fb_terms': best_t, 'fb_docs': best_d, 'fb_lambda': best_w}
    print(f'  Best CV RM3 for {level}: fb_terms={best_t}, fb_docs={best_d}, '
          f'fb_lambda={best_w} -> mean held-out MAP={param_cv_scores[(best_t,best_d,best_w)]:.4f}')

df_rm3_tuning = pd.DataFrame(rm3_tuning_records)
print('\nFull RM3 CV tuning results:')
display(df_rm3_tuning.sort_values(['level', 'cv_mean_map'], ascending=[True, False]))



CV-tuning RM3 for verbosity level: title
  Best CV RM3 for title: fb_terms=20, fb_docs=10, fb_lambda=0.5 -> mean held-out MAP=0.1939

CV-tuning RM3 for verbosity level: title_desc
  Best CV RM3 for title_desc: fb_terms=20, fb_docs=10, fb_lambda=0.7 -> mean held-out MAP=0.2063

CV-tuning RM3 for verbosity level: title_desc_narr
  Best CV RM3 for title_desc_narr: fb_terms=20, fb_docs=10, fb_lambda=0.7 -> mean held-out MAP=0.2167

Full RM3 CV tuning results:


,level,fb_terms,fb_docs,fb_lambda,cv_mean_map,cv_std_map
25,title,20,10,0.5,0.193875,0.035399
19,title,20,3,0.5,0.191942,0.036590
26,title,20,10,0.7,0.191751,0.034210
22,title,20,5,0.5,0.191213,0.036348
24,title,20,10,0.3,0.189866,0.032822
...,...,...,...,...,...,...
55,title_desc_narr,5,3,0.5,0.162039,0.041824
63,title_desc_narr,10,3,0.3,0.161684,0.045262
60,title_desc_narr,5,10,0.3,0.154583,0.043665
57,title_desc_narr,5,5,0.3,0.150479,0.044881


## 10. Build Final Tuned BM25 + RM3 Pipelines

In [21]:
tuned_rm3_pipe = {}
for level, params in best_rm3_params.items():
    bm25_params = best_bm25_params[level]
    base = tuned_bm25[level]
    rm3 = pt.rewrite.RM3(
        index,
        fb_terms=params['fb_terms'],
        fb_docs=params['fb_docs'],
        fb_lambda=params['fb_lambda']  # ← changed
    )
    retriever2 = pt.terrier.Retriever(
        index, wmodel='BM25', num_results=1000,
        controls={'bm25.b': bm25_params['b'], 'bm25.k_1': bm25_params['k1']}
    )
    tuned_rm3_pipe[level] = base >> rm3 >> retriever2
    print(f'RM3 pipeline [{level}] ready. '
          f'BM25 b={bm25_params["b"]} k1={bm25_params["k1"]} | '
          f'RM3 fb_terms={params["fb_terms"]} fb_docs={params["fb_docs"]} fb_lambda={params["fb_lambda"]}')  # ← changed

RM3 pipeline [title] ready. BM25 b=0.5 k1=1.2 | RM3 fb_terms=20 fb_docs=10 fb_lambda=0.5
RM3 pipeline [title_desc] ready. BM25 b=0.5 k1=1.5 | RM3 fb_terms=20 fb_docs=10 fb_lambda=0.7
RM3 pipeline [title_desc_narr] ready. BM25 b=0.75 k1=1.5 | RM3 fb_terms=20 fb_docs=10 fb_lambda=0.7


## 11. Final Evaluation — BM25 vs BM25+RM3 per Verbosity Level

Parameters were selected via 5-fold CV on held-out query subsets.
The final evaluation here uses **all 50 queries**, which is correct because
the parameter choice was made without seeing the full query set together.


In [23]:
all_results = []

for level, topics_df in verbosity_levels.items():
    print(f"\n{'='*60}")
    print(f'Verbosity level: {level.upper()}')
    print(f'  BM25 CV params  : b={best_bm25_params[level]["b"]}, k1={best_bm25_params[level]["k1"]}')
    print(f'  RM3  CV params  : {best_rm3_params[level]}')
    print(f"{'='*60}")

    res = pt.Experiment(
        retr_systems=[
            bm25,                    # BM25 default
            tuned_bm25[level],       # BM25 tuned (CV)
            tuned_rm3_pipe[level],   # BM25 tuned + RM3 tuned (CV)
        ],
        topics=topics_df,
        qrels=qrels,
        eval_metrics=eval_metrics,
        names=['BM25_default', 'BM25_tuned_CV', 'BM25_tuned+RM3_CV'],
        baseline=0
    )
    res.insert(0, 'verbosity', level)
    all_results.append(res)
    display(res)

df_all_results = pd.concat(all_results, ignore_index=True)
print('\nCombined results table (CV-tuned parameters):')
display(df_all_results)



Verbosity level: TITLE
  BM25 CV params  : b=0.5, k1=1.2
  RM3  CV params  : {'fb_terms': 20, 'fb_docs': 10, 'fb_lambda': 0.5}


,verbosity,name,map,P_10,recall_1000,ndcg_cut_10,map +,map -,map p-value,P_10 +,P_10 -,P_10 p-value,recall_1000 +,recall_1000 -,recall_1000 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,title,BM25_default,0.170642,0.530,0.356296,0.463887,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,title,BM25_tuned_CV,0.178687,0.556,0.359317,0.484830,38.0,12.0,0.004401,15.0,7.0,0.062609,30.0,12.0,0.325814,24.0,17.0,0.042786
2,title,BM25_tuned+RM3_CV,0.193875,0.522,0.378097,0.461774,36.0,14.0,0.001080,16.0,14.0,0.704410,32.0,13.0,0.006474,21.0,21.0,0.882098



Verbosity level: TITLE_DESC
  BM25 CV params  : b=0.5, k1=1.5
  RM3  CV params  : {'fb_terms': 20, 'fb_docs': 10, 'fb_lambda': 0.7}


,verbosity,name,map,P_10,recall_1000,ndcg_cut_10,map +,map -,map p-value,P_10 +,P_10 -,P_10 p-value,recall_1000 +,recall_1000 -,recall_1000 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,title_desc,BM25_default,0.191874,0.534,0.396070,0.447932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,title_desc,BM25_tuned_CV,0.196161,0.568,0.399586,0.481851,32.0,18.0,0.041556,16.0,5.0,0.006627,28.0,15.0,0.143130,30.0,16.0,0.002941
2,title_desc,BM25_tuned+RM3_CV,0.206282,0.564,0.412712,0.481288,32.0,18.0,0.003098,22.0,11.0,0.083199,35.0,14.0,0.001018,30.0,16.0,0.028708



Verbosity level: TITLE_DESC_NARR
  BM25 CV params  : b=0.75, k1=1.5
  RM3  CV params  : {'fb_terms': 20, 'fb_docs': 10, 'fb_lambda': 0.7}


,verbosity,name,map,P_10,recall_1000,ndcg_cut_10,map +,map -,map p-value,P_10 +,P_10 -,P_10 p-value,recall_1000 +,recall_1000 -,recall_1000 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,title_desc_narr,BM25_default,0.201098,0.612,0.408935,0.525577,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,title_desc_narr,BM25_tuned_CV,0.202311,0.608,0.410648,0.529981,29.0,21.0,0.139340,5.0,5.0,0.621960,24.0,17.0,0.097099,16.0,13.0,0.509439
2,title_desc_narr,BM25_tuned+RM3_CV,0.216684,0.588,0.426496,0.513708,29.0,21.0,0.028517,12.0,15.0,0.233829,26.0,22.0,0.037298,18.0,24.0,0.510271



Combined results table (CV-tuned parameters):


,verbosity,name,map,P_10,recall_1000,ndcg_cut_10,map +,map -,map p-value,P_10 +,P_10 -,P_10 p-value,recall_1000 +,recall_1000 -,recall_1000 p-value,ndcg_cut_10 +,ndcg_cut_10 -,ndcg_cut_10 p-value
0,title,BM25_default,0.170642,0.530,0.356296,0.463887,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,title,BM25_tuned_CV,0.178687,0.556,0.359317,0.484830,38.0,12.0,0.004401,15.0,7.0,0.062609,30.0,12.0,0.325814,24.0,17.0,0.042786
2,title,BM25_tuned+RM3_CV,0.193875,0.522,0.378097,0.461774,36.0,14.0,0.001080,16.0,14.0,0.704410,32.0,13.0,0.006474,21.0,21.0,0.882098
3,title_desc,BM25_default,0.191874,0.534,0.396070,0.447932,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,title_desc,BM25_tuned_CV,0.196161,0.568,0.399586,0.481851,32.0,18.0,0.041556,16.0,5.0,0.006627,28.0,15.0,0.143130,30.0,16.0,0.002941
5,title_desc,BM25_tuned+RM3_CV,0.206282,0.564,0.412712,0.481288,32.0,18.0,0.003098,22.0,11.0,0.083199,35.0,14.0,0.001018,30.0,16.0,0.028708
6,title_desc_narr,BM25_default,0.201098,0.612,0.408935,0.525577,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,title_desc_narr,BM25_tuned_CV,0.202311,0.608,0.410648,0.529981,29.0,21.0,0.139340,5.0,5.0,0.621960,24.0,17.0,0.097099,16.0,13.0,0.509439
8,title_desc_narr,BM25_tuned+RM3_CV,0.216684,0.588,0.426496,0.513708,29.0,21.0,0.028517,12.0,15.0,0.233829,26.0,22.0,0.037298,18.0,24.0,0.510271


## 12. Save Results

In [25]:
os.makedirs('results', exist_ok=True)

df_all_results.to_csv('results/final_results_cv.csv', index=False)
df_tuning.to_csv('results/bm25_cv_tuning.csv', index=False)
df_rm3_tuning.to_csv('results/rm3_cv_tuning.csv', index=False)

print('Saved:')
print('  results/final_results_cv.csv')
print('  results/bm25_cv_tuning.csv')
print('  results/rm3_cv_tuning.csv')


Saved:
  results/final_results_cv.csv
  results/bm25_cv_tuning.csv
  results/rm3_cv_tuning.csv
